# Writing a workflow for the framework: specs and bindings

This notebook is for the authors of workflow packages, such as esssans.
A package that offers a workflow to the framework writes two things, both against `ess.spec`:

- a **spec**: the workflow's name, version, and three pydantic models: its params, its inputs, and its outputs. Apps build forms from it, the framework checks requests against it, and records name it. It is all that anyone outside the package sees.
- a **binding**: the code that computes the spec. Apart from the validators of the models and of the types in them, it is the only code of the package that the backend runs.

`ess.spec` needs pydantic, and scipp for specs whose data are scipp objects; `ess.spec.pipeline` adds sciline.
A workflow package depends on `ess.spec`, not on `ess.dispatch`, the client and backend that run it.

The thread: a reduction of one run, its spec and binding, then the same reduction over several runs, summed as they arrive by a *driver*.
At each step, what the framework gives the binding, and what the binding's author promises in return.

## 0. The workflow you have: a sciline pipeline

The toy SANS reduction of `toy_sans.py` stands in for esssans.
A run's file holds its counts per pixel and its monitor count; a sample run's file also holds the sample's transmission.
`load` stands for loading and correcting a run, the costly step.
The reduction sums the pixels into Q bins (the numerator), divides by the monitor count times the transmission (the denominator), and subtracts the can's curve from the sample's.

The pipeline's keys for what a user gives are types that a spec can declare, defined in `toy_sans.py` as:

```python
SampleFile = NewType('SampleFile', Path)   # a sample run's file
CanFile = NewType('CanFile', Path)         # an empty-can run's file
IofQ = NewType('IofQ', sc.Variable)        # the result

class Reduction(BaseModel):
    bins: int = Field(default=2, gt=0)

class Transmission(BaseModel):
    value: float = Field(gt=0, le=1)
```

`toy_sans.write_runs` writes the toy runs as small files:

In [ ]:
import tempfile
from pathlib import Path

import toy_sans
from toy_sans import CanFile, IofQ, Reduction, SampleFile, Transmission, make_pipeline

folder = tempfile.TemporaryDirectory()
paths = toy_sans.write_runs(Path(folder.name))
print(paths[101].read_text())

workflow = make_pipeline({})
workflow[SampleFile] = paths[101]
workflow[CanFile] = paths[110]
workflow[Reduction] = Reduction(bins=2)
workflow[Transmission | None] = None  # take the transmission in the sample's file
workflow.compute(IofQ)

## 1. The spec of one run

The spec declares three models:

| Model | Holds | In a request | When the workflow runs |
|---|---|---|---|
| params | configuration: literals only | the values | the same values |
| inputs | data only | a reference per value | the data the references name |
| outputs | results | (none) | the data the workflow returns |

A *literal* is a value with a JSON form that the client knows: a number, a string, a model of such values.
A *data value* is one the backend fetches: a file, an array, a model holding one, or any other output of a record.
Each field of the params model is a *parameter*, and is a *group*, a model of literals, or a list of groups.
Each field of the inputs model is an *input*, and is a data value, possibly optional, or a list of data values.

In [ ]:
from pydantic import BaseModel, ConfigDict, Field

from ess.spec import WorkflowSpec


class IofQParams(BaseModel):
    reduction: Reduction = Field(default_factory=Reduction)
    transmission: Transmission | None = Field(
        default=None, description="The sample's; None takes the one in its file."
    )


class IofQInputs(BaseModel):
    sample: SampleFile
    can: CanFile


class IofQOutputs(BaseModel):
    model_config = ConfigDict(arbitrary_types_allowed=True)

    iofq: IofQ


IOFQ = WorkflowSpec(
    name='toy-sans-iofq',
    version=1,
    title='I(Q)',
    description='I(Q) of one sample run, minus that of one can run.',
    params=IofQParams,
    inputs=IofQInputs,
    outputs=IofQOutputs,
)

Rules of the spec, each with its reason:

- **An input holds data, given by reference.** A request names a dataset or another record's output; the binding gets the data read, a file as a local path. So the spec says what the data is, by its type (`SampleFile`, `IofQ`), never where it lives. A file is a `NewType` of `Path`.
- **Each field has a type of its own.** The type is the field's key in a sciline pipeline, and the framework checks references by it. `SampleFile` and `CanFile` are both paths, so each is a `NewType`. `WorkflowSpec` refuses two fields of one type.
- **Defaults live in the models only.** The binding gets every value filled in, so a default in the binding would never be used. The default of an input is `None` or an empty list.
- **A check of one value belongs to its type; a check of several is a model validator.** `Reduction` refuses `bins=0`, and the params and inputs models have no field validators. A template checks each fixed value by its own field, before the others are known ([README, Templates](../README.md#templates)).
- **Name and version identify the spec.** Records name them. `version` is the version of the interface: it changes with any change a consumer could observe, in the params, inputs, or outputs models, or in what the workflow computes. The optional field `code_revision` names the code itself; records do not carry it yet.

An app reads the spec, not the package's code.
`serialize()` gives the JSON Schema of the params, from which a form is built, and for each input the type of data it takes, from which a picker of datasets or records is built:

In [ ]:
serialized = IOFQ.serialize()
print(list(serialized.params_schema['properties']))
serialized.inputs_schema['properties']

A request lives in two phases.
In the *request phase*, before any data is read, each input is a reference, and the framework checks that the data it names fit the input's type.
In the *call phase*, the backend reads the data and builds the inputs model from them.
Params are the same in both phases, so the params model refuses what cannot run before anything runs; the inputs model takes data only, never a reference:

In [ ]:
from pydantic import ValidationError

from ess.spec import DatasetRef

try:
    IofQParams(reduction={'bins': 0})
except ValidationError as error:
    print(error)
try:
    IofQInputs(sample=DatasetRef(dataset='run:101'), can=paths[110])
except ValidationError as error:
    print(error)

## 2. The binding of one run

A binding is what the backend calls to compute a spec's requests.
The simplest binding is a function.
It takes an instance of the params model and one of the inputs model, and returns one of the outputs model.
It gets every default filled in and every input read:

In [ ]:
from ess.spec import function_binding


def iofq(params: IofQParams, inputs: IofQInputs) -> IofQOutputs:
    workflow = make_pipeline({})
    workflow[Reduction] = params.reduction
    workflow[Transmission | None] = params.transmission
    workflow[SampleFile] = inputs.sample
    workflow[CanFile] = inputs.can
    return IofQOutputs(iofq=workflow.compute(IofQ))


draft = function_binding(IOFQ, iofq)
params = IofQParams()
inputs = IofQInputs(sample=paths[101], can=paths[110])
draft.compute(params, inputs)

A request with every value given, a *plain request*, goes through `compute(params, inputs)`.
The spec is generic over its models, so a type checker checks `iofq` against `IOFQ`.

For a sciline pipeline, `PipelineBinding` does the same.
Each parameter and input sets the key of its type as written, here `Reduction`, `Transmission | None`, `SampleFile`, and `CanFile`, and each output is computed as the key of its type, here `IofQ`.
`make_pipeline` gets the fixed values by name, so that the graph may depend on them, and returns a pipeline of its own at each call:

In [ ]:
from ess.spec.pipeline import PipelineBinding

single = PipelineBinding(IOFQ, make_pipeline)
single.compute(params, inputs)

A pipeline with keys of its own names them with `keys=`, such as esssans with `keys={'sample': Filename[SampleRun], 'iofq': BackgroundSubtractedIofQ}`.

Each binding takes its spec, and the backend finds a request's binding by the spec's name and version.

**Promise: a binding modifies none of its inputs, and returns no output that shares memory with an input.**
The framework passes the values it holds without a copy, such as another record's output that a request references.
A binding that modified one, or returned a view of one, would change another record's value.
`forward` in `toy_sans.py` returns `inputs.curve['Q', 0].copy()` for this reason.

## 3. Stages: compute the fixed part once

When a user tunes `bins`, the runs stay the same.
The client then keeps a *stage*: a template whose fixed values stay, with *blanks*, the parameters and inputs left open (see [client-authors.ipynb](client-authors.ipynb)).
The backend calls `make_stage(fixed, blanks)` once, and the returned callable once per call through the stage, with the values of the blanks.
Both take values by name: a parameter as its group, an input as its data.
A parameter or input in neither, here `transmission`, takes its default.
`PipelineBinding` cuts the pipeline at the keys of the blanks with `sciline.Stage`, and computes what does not depend on them once:

In [ ]:
fixed = {'sample': paths[101], 'can': paths[110]}
call = single.make_stage(fixed, ('reduction',))
toy_sans.LOADED.clear()
for bins in (1, 2, 4):
    print(bins, call({'reduction': Reduction(bins=bins)}).iofq.values)
print('runs loaded:', len(toy_sans.LOADED))

Two loads for three calls: the sample run and the can run, once each.
A function binding has no `make_stage`, since a stage over a function would hold nothing; the backend refuses a stage over it when the stage is made.

**Promise: a call through a stage gives the outputs of the plain request** (the *caching* symmetry).
The framework records a call through a stage as its plain request, so the record must not depend on how it was computed.
The `sciline.Stage` keeps this promise, given a pure pipeline.
`ess.spec.testing` checks it by computing both, and also checks that no output shares memory with an input:

In [ ]:
from ess.spec.testing import check_caching

check_caching(single, fixed, [{'reduction': Reduction(bins=n)} for n in (2, 1, 2)])

## 4. The spec of several runs: tables

Many reductions combine runs: here, several sample runs and several can runs, summed.
A *table* is a set of list fields of equal length.
Its *index* is the field that names it, such as the sample runs; its other *columns* are attached to the index, such as the transmission of each sample run.
Each position is a *row*.
`tabulate` makes the multi-run spec from the single-run spec by naming the index of each table, with the columns attached to it:

In [ ]:
IOFQ_MULTI = IOFQ.tabulate(
    name='toy-sans-iofq-multi',
    version=1,
    tables={'sample': ('transmission',), 'can': ()},
    title='I(Q) of summed runs',
    description='I(Q) of the summed sample runs, minus that of the summed can runs.',
)
for name, field in IOFQ_MULTI.params.model_fields.items():
    print(f'{name}: {field.annotation}')
IOFQ_MULTI.serialize().inputs_schema['properties']

Rules of tables:

- **Each field named becomes a list on its own side.** An index loses `| None`: `sample: SampleFile` becomes `list[SampleFile]` with at least one row. An attached column keeps its type as written: `transmission: Transmission | None` becomes `list[Transmission | None]`, and a row given without one has `None`. The other fields, here `reduction`, and the outputs model are those of `IOFQ`.
- **A row holds what belongs to one run.** A value that differs per run is in the run's file; a value that the file lacks or has wrong, such as a transmission, is a column attached to the run, so the request and its record hold it. Tables are independent of each other; which runs go together is the app's choice.
- **The index values of a table are distinct**, as in a pandas DataFrame: a run is in a table once.
- **Structure below a row is the binding's**, such as detector banks. The framework sees rows only.
- **The models that `tabulate` makes have no static type.** A client gives the params of `IOFQ_MULTI` as a dict, and a driver gets columns.
- **A spec without a single-run form declares its tables itself**, such as a stitch of all angles: `WorkflowSpec(..., tables={'angle': ()})`. A list field makes a table only where `tables` names it.

## 5. A driver: summing runs as they arrive

When runs arrive one at a time, a user wants the sum so far after each, without reducing the earlier runs again.
The client's *accumulator* takes one *push* at a time, each one or more rows of one table (see [client-authors.ipynb](client-authors.ipynb)).
Behind it is a *driver*: an object with `push(table, columns)` and `outputs()`, which the workflow package writes for one spec.
A push names its table by its index, and gives the index and each column attached to it, by name, as lists of equal length: a parameter's values as groups, an input's as data.
Only a driver accumulates: the framework never decides how rows combine.

This driver sums each run's numerator and denominator per table, and computes I(Q) once from the sums.
It cuts the one-run pipeline with `sciline.Stage` twice: at the keys of a table's columns, to compute one run's numerator and denominator, and at the four sums, to compute I(Q).
It gets the fixed values that are not tables, here `reduction`, when it is made:

In [ ]:
from collections.abc import Mapping, Sequence
from typing import Any

import sciline
import scipp as sc

from toy_sans import CanRun, Denominator, Numerator, SampleRun

KEYS = {'sample': SampleFile, 'transmission': Transmission | None, 'can': CanFile}


class IofQDriver:
    def __init__(self, fixed: Mapping[str, Any]) -> None:
        workflow = make_pipeline(fixed)
        workflow[Reduction] = fixed['reduction']
        self._contribute = {
            index: sciline.Stage(
                workflow,
                inputs=[KEYS[name] for name in IOFQ_MULTI.columns(index)],
                outputs=(Numerator[run], Denominator[run]),
            )
            for index, run in (('sample', SampleRun), ('can', CanRun))
        }
        sums = [
            key[run] for key in (Numerator, Denominator) for run in (SampleRun, CanRun)
        ]
        self._finalize = sciline.Stage(workflow, inputs=sums, outputs=(IofQ,))
        self._sums: dict[str, dict[Any, sc.Variable]] = {}  # per table, by key

    def push(self, table: str, columns: Mapping[str, Sequence[Any]]) -> None:
        stage = self._contribute[table]
        for row in zip(*columns.values(), strict=True):
            keys = (KEYS[name] for name in columns)
            parts = stage.compute(dict(zip(keys, row, strict=True)))
            old = self._sums.get(table)
            self._sums[table] = (
                parts if old is None else {k: old[k] + v for k, v in parts.items()}
            )

    def outputs(self) -> IofQOutputs:
        if empty := sorted(self._contribute.keys() - self._sums.keys()):
            raise ValueError(f'no rows in {empty} yet')
        sums = {k: v for parts in self._sums.values() for k, v in parts.items()}
        return IofQOutputs(iofq=self._finalize.compute(sums)[IofQ])


driver = IofQDriver({'reduction': Reduction(bins=2)})
toy_sans.LOADED.clear()
driver.push('sample', {'sample': [paths[101]], 'transmission': [None]})
driver.push('can', {'can': [paths[110]]})
print(driver.outputs().iofq.values)
driver.push(
    'sample',
    {
        'sample': [paths[102], paths[103]],
        'transmission': [None, Transmission(value=0.6)],
    },
)
print(driver.outputs().iofq.values)
print('runs loaded:', len(toy_sans.LOADED))

Four loads for four runs: a push loads only its own rows.
Summing numerators and denominators apart, and dividing once, is what makes a sum of runs correct; a sum of normalized curves would not be.
The driver needs no spec, backend, or client, so a package uses and tests it as plain code.

The author of a driver promises, and the backend does not check:

- **`outputs()` gives the outputs of every row pushed so far**, however the rows were grouped into pushes, and whatever the order of pushes into different tables (the *arrival* and *order* symmetries). So each sum may depend on the fixed values and on one table only.
- **Within a table, the rows keep the order in which they were pushed.**
- **`push` does not modify a value it is given.** A value may be another record's output. A driver that adds in place copies the first contribution, so that adding never changes a value that another record returned. `IofQDriver` adds with `a + b`, which modifies nothing.
- **A driver may refuse a state**, such as one with no can row yet, by raising in `outputs()`. Later pushes may make the state valid.
- **Model validators change no value.** A driver gets each value checked by its own type only: the params model's validators run on the params of a state when it is read, and the inputs model's do not run for an accumulator.

In return, `outputs()` may share memory with what the driver holds: the backend is done with the outputs of a state before it pushes the next rows.

`check_arrival_and_order` takes the function that makes a driver, the fixed values that are not tables, and the columns of each table by its index.
It pushes the rows in several schedules, one row at a time and in groups, table by table and alternating, and reads between pushes.
It compares each state with what a new driver gives after one push per table, skipping states that the driver refuses.
It also fails if a push modifies the values it was given:

In [ ]:
from ess.spec.testing import check_arrival_and_order

tables = {
    'sample': {
        'sample': [paths[n] for n in (101, 102, 103)],
        'transmission': [None, None, Transmission(value=0.6)],
    },
    'can': {'can': [paths[n] for n in (110, 111)]},
}
check_arrival_and_order(IofQDriver, {'reduction': Reduction(bins=2)}, tables)

Both sides of the comparison go through the driver, so the check does not prove the outputs right.

A sum that depends on two tables would break the order symmetry: the can's sum would change with the sample rows pushed before it.
esssans has this case, since its pixel masks come from the sample run.
Its driver must give outputs that do not depend on the order of pushes, for example by keeping the can rows that a mask applies to; `check_arrival_and_order` catches a driver whose outputs do.

A workflow that needs every row at once, such as a fit over all runs, writes a driver that keeps its rows and computes over all of them in `outputs()`.
The outputs are the same as for a driver that sums; only the cost differs, and which to pay is the workflow's choice.

## 6. The binding of several runs

`DriverBinding` binds the multi-run spec to the function that makes a driver.
A plain request makes a driver, pushes the columns of each table in one push, and reads the outputs:

In [ ]:
from ess.spec import DriverBinding

multi = DriverBinding(IOFQ_MULTI, IofQDriver)
multi.compute(
    IOFQ_MULTI.params(transmission=tables['sample']['transmission']),
    IOFQ_MULTI.inputs(sample=tables['sample']['sample'], can=tables['can']['can']),
)

For an accumulator, the backend calls `make_accumulator(fixed, blanks)`, with the blanks named by their indexes.
The binding makes a driver from the fixed values that are not tables, and pushes the columns of each table in `fixed`, such as a can run fixed by the client's template.
A fixed table comes under its index's name, as its columns, as a push gives them.
The backend then pushes each client push into the driver:

In [ ]:
driver = multi.make_accumulator(
    {'reduction': Reduction(bins=2), 'can': {'can': [paths[110]]}}, ('sample',)
)
driver.push('sample', {'sample': [paths[101]], 'transmission': [None]})
driver.outputs()

**Promise: with one row in each table, the binding of the multi-run spec gives the outputs of the single-run spec** (the *one-row* symmetry).
`tabulate` makes the specs agree: with one row in each table whose index is given, and none where it is left out, a request of `IOFQ_MULTI` is a request of `IOFQ`.
`one_row` makes that request from a request of `IOFQ`:

In [ ]:
from ess.spec.testing import check_one_row, one_row

params = IofQParams(transmission=Transmission(value=0.6))
one_row(IOFQ_MULTI, params, inputs)

The package keeps the promise: the driver must compute what the pipeline computes.
`check_one_row` computes both and compares them; it is the only check against an independent reference:

In [ ]:
check_one_row(single, multi, params, inputs)

Each binding serves some of the client's actions, and the backend refuses the others when a stage or an accumulator is made:

| Binding | Plain request | Stage | Accumulator |
|---|---|---|---|
| `PipelineBinding` | `compute` | `make_stage` | refused |
| `DriverBinding` | `compute` | refused | `make_accumulator` |
| `function_binding` | `compute` | refused | refused |

## 7. What the framework gives the binding

The binding may rely on this:

| Call | When, and with what |
|---|---|
| `compute(params, inputs)` | once per plain request; instances of the params and inputs models, with data read and defaults filled in; possibly from several threads at once |
| `make_stage(fixed, blanks)` | once per stage of a client, at its first call; every value but the blanks, by name; possibly from several threads at once |
| the callable it returns | once per call through the stage, with the values of the blanks, by name; possibly from several threads at once |
| `make_accumulator(fixed, blanks)` | once per accumulator; every value but the tables that the pushes fill, by name. `DriverBinding` calls its function that makes a driver here, and in `compute`, possibly from several threads at once |
| `driver.push(table, columns)` | once per client push, with the index and every attached column, as lists of equal length, data read; in the order logged; one thread at a time; never while the readers of the state before it are busy |
| `driver.outputs()` | once per state that someone reads; one thread at a time |

Each value the binding gets holds the type its field declares: the backend checks each reference's fit by type when a request is made, and each value by its type when it is read.
[framework-promises.ipynb](framework-promises.ipynb) traces these calls for one accumulator.

## Summary: what the author of a workflow promises

| Promise | Checked by |
|---|---|
| each parameter is a group, each input a data value, and each field has a type of its own | `WorkflowSpec` |
| defaults live in the models only | nobody |
| a check of one value belongs to its type; a check of several is a model validator, and changes no value | nobody |
| a new version when the params, inputs, or outputs models change | nobody |
| the binding returns no output that shares memory with an input | `check_one_row`, `check_caching` (numpy-backed values) |
| the binding modifies no input | nobody, except for the values a push gets (`check_arrival_and_order`) |
| the pipeline is pure, so a call through a stage gives the outputs of the plain request | `check_caching` |
| the multi-run spec with one row per table gives the single-run spec's outputs | `check_one_row` |
| after each push, the driver's outputs are those of all rows pushed so far, in any grouping and in any order across tables | `check_arrival_and_order` |
| `push` modifies no value | `check_arrival_and_order` |
| structure below a row belongs to the binding | nobody |

A package registers its bindings through the entry-point group `ess.dispatch.workflows` (decided, not implemented).
Until then, a notebook or app passes them to `local(proposal=..., datasets=..., bind=[single, multi, forward_binding])`.

In [ ]:
folder.cleanup()